In [1]:
import pandas as pd
df = pd.read_csv("YoutubeCommentsDataSet.csv")
print(df.shape)
print(df.isna().sum())
print("duplicates:", df.duplicated().sum())
print(df["sentiment"].value_counts())
print(df["comment"].str.split().str.len().describe())
df.sample(10, random_state=1)

(18408, 2)
Comment      44
Sentiment     0
dtype: int64
duplicates: 531


KeyError: 'sentiment'

In [2]:
import pandas as pd
df = pd.read_csv("YoutubeCommentsDataSet.csv")
print(df.shape)
print(df.isna().sum())
print("duplicates:", df.duplicated().sum())
print(df["Sentiment"].value_counts())
print(df["Comment"].str.split().str.len().describe())
df.sample(10, random_state=1)

(18408, 2)
Comment      44
Sentiment     0
dtype: int64
duplicates: 531
Sentiment
positive    11432
neutral      4638
negative     2338
Name: count, dtype: int64
count    18364.000000
mean        32.932640
std         45.293523
min          1.000000
25%         12.000000
50%         21.000000
75%         38.000000
max       1353.000000
Name: Comment, dtype: float64


,Comment,Sentiment
4485,bong cloud has one variation super easy to lea...,positive
10394,thật sự mình khuyên mono nên nghe lời khuyên c...,neutral
14310,i wish i had seen this years ago trying to vis...,positive
17290,im sure everyone would love to hear about some...,neutral
2811,never thought i would enjoy a 6 hour programmi...,positive
1544,man city playing so well so far but i feel lik...,neutral
2319,hes spot on with that maintenance number of 20...,positive
12134,this was so informative im in icse board and i...,positive
13185,what a honor to have this man in our justice h...,positive
17706,when your game turned pink after the render pi...,neutral


In [3]:
%pip install langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------------------------------------- 981.5/981.5 kB 11.6 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993364 sha256=343895b76285b1fd6a808803a8fa358ea6a08d1f6c73a8b38365cd8c3c8cef93
  Stored in directory: c:\users\giova\appdata\local\pip\cache\wheels\c1\67\88\e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.


In [4]:
df.columns = df.columns.str.strip().str.lower()
df = df.dropna(subset=["comment"]).drop_duplicates(subset="comment")
print("after dedupe:", df.shape)

from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0          # makes results reproducible

def lang(t):
    try:
        return detect(t)
    except Exception:
        return "unk"

df["lang"] = df["comment"].apply(lang)
print(df["lang"].value_counts().head(10))

df = df[df["lang"] == "en"].copy()
print(df["sentiment"].value_counts())

after dedupe: (17871, 2)
lang
en    15880
es      363
pt      360
fr      135
vi      106
id       69
tl       65
de       63
it       61
pl       57
Name: count, dtype: int64
sentiment
positive    10338
neutral      3263
negative     2279
Name: count, dtype: int64


In [5]:
import re, nltk, numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords"); nltk.download("wordnet")

stop = set(stopwords.words("english")) | {
    "video", "youtube", "channel", "like", "get", "would", "one",
    "really", "also", "go", "make", "even", "much", "thing"}
lem = WordNetLemmatizer()

def clean(t):
    t = re.sub(r"http\S+|www\.\S+|&\w+;", " ", t.lower())
    t = re.sub(r"[^a-z\s]", " ", t)               # removes digits, emojis, punctuation
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)          # sooooo -> soo
    return " ".join(lem.lemmatize(w) for w in t.split()
                    if w not in stop and len(w) > 2)

df["clean"] = df["comment"].apply(clean)
df = df[df["clean"].str.split().str.len() >= 3].reset_index(drop=True)

print(df["sentiment"].value_counts())
df[["comment", "clean"]].sample(5, random_state=2)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\giova\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\giova\AppData\Roaming\nltk_data...


sentiment
positive    10118
neutral      3179
negative     2254
Name: count, dtype: int64


,comment,clean
13971,one of the first films we saw as a family at t...,first film saw family drivein memory think ame...
6889,greg is so high hes on another planet,greg high he another planet
10197,sad but no judgement was he wrong hell yeah bu...,sad judgement wrong hell yeah without enablers...
8507,sting returning at wm 31 and rock returning at...,sting returning rock returning raw litrerly lo...
3199,great platform for auto crypto arbitrage but i...,great platform auto crypto arbitrage think spe...


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_df=0.5, min_df=5, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(df["clean"])

density = X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])
print("TF-IDF shape:", X_tfidf.shape, "| non-zero share:", f"{density:.3%}")

TF-IDF shape: (15551, 8368) | non-zero share: 0.181%


In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
X_emb = model.encode(df["comment"].tolist(), batch_size=64, show_progress_bar=True)
print("Embedding shape:", X_emb.shape)

C:\Users\giova\nlp-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 243/243 [00:35<00:00,  6.85it/s]

Embedding shape: (15551, 384)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

i = df.index[df["sentiment"] == "negative"][0]
print("QUERY:", df.loc[i, "comment"][:200], "\n")

for name, X in [("TF-IDF", X_tfidf), ("SBERT", X_emb)]:
    sims = cosine_similarity(X[i], X).ravel()
    top = sims.argsort()[::-1][1:4]
    print(f"--- {name} nearest comments ---")
    for j in top:
        print(f"{sims[j]:.2f} | {df.loc[j, 'comment'][:150]}")

QUERY: here in nz 50 of retailers don’t even have contactless credit card machines like paywave which support apple pay they don’t like the high fees that come with these 

--- TF-IDF nearest comments ---
0.72 | for now i need both apple pay and the physical credit card
0.52 | apple pay is so convenient secure and easy to use i used it while at the korean and japanese airports no need for physical credit cards
0.42 | at this point i don’t think his credit card company cares anymore


ValueError: Expected 2D array, got 1D array instead:
array=[ 6.22806326e-03 -1.45612081e-04  4.08999473e-02  1.96810178e-02
  6.63703009e-02 -1.02621166e-03  2.35444419e-02 -4.25168611e-02
  3.99537161e-02  2.10211370e-02  5.12438603e-02 -1.45149534e-03
 -3.61188836e-02  4.44699451e-03  8.85263011e-02 -3.80537212e-02
 -2.59062089e-02 -1.50786698e-01  1.27846317e-03  9.43625867e-02
 -2.79835984e-02 -9.35591664e-03  2.97912881e-02  1.34012578e-02
  1.52165527e-02 -1.39754005e-02 -4.91332896e-02  7.10007399e-02
 -6.75223954e-03  2.71384344e-02 -4.63445559e-02  5.50023839e-02
  7.46061876e-02  3.03032361e-02  3.70637961e-02 -6.56022951e-02
  1.45672020e-02  4.74445932e-02 -1.37590230e-01 -5.31272329e-02
  2.07242873e-02  5.30268922e-02 -4.21417840e-02  2.51926538e-02
  7.97538534e-02  1.05005235e-03 -4.94569587e-03  3.40029597e-02
 -4.89295125e-02  7.56827518e-02  6.06611138e-03 -5.56397885e-02
 -2.26774835e-03 -2.83596609e-02 -7.97958449e-02 -3.51765454e-02
 -4.75373492e-03  2.36011613e-0

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

i = df.index[df["sentiment"] == "negative"][0]
print("QUERY:", df.loc[i, "comment"][:200], "\n")

for name, X in [("TF-IDF", X_tfidf), ("SBERT", X_emb)]:
    sims = cosine_similarity(X[i:i+1], X).ravel()
    top = sims.argsort()[::-1][1:4]
    print(f"--- {name} nearest comments ---")
    for j in top:
        print(f"{sims[j]:.2f} | {df.loc[j, 'comment'][:150]}")

QUERY: here in nz 50 of retailers don’t even have contactless credit card machines like paywave which support apple pay they don’t like the high fees that come with these 

--- TF-IDF nearest comments ---
0.72 | for now i need both apple pay and the physical credit card
0.52 | apple pay is so convenient secure and easy to use i used it while at the korean and japanese airports no need for physical credit cards
0.42 | at this point i don’t think his credit card company cares anymore
--- SBERT nearest comments ---
0.71 | in the united states we have an abundance of retailers that accept apple pay but its still good to carry the physical card with you being there are st
0.67 | whenever i go to a place that doesn’t take apple pay doesn’t happen too often it’s such a drag between ‘contactless covid’ habits and my getting the a
0.67 | apple pay is so convenient secure and easy to use i used it while at the korean and japanese airports no need for physical credit cards
